In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from pathlib import Path
import torch
from transformers import BartTokenizer, BartForConditionalGeneration
from torch.utils.data import DataLoader
from torch import nn
from torch.functional import F
from tqdm import tqdm
from transformers import DataCollatorForSeq2Seq
from torch.utils.data import DataLoader
from datasets import Dataset
import numpy as np

from src.data_utils import get_sample_from_row_original, filter_irrelevant, prune_frequent_samples
from src.inference_utils import predict
from src.metrics import lemmatization_accuracy

In [2]:
CURDIR = Path.cwd()

DATADIR = CURDIR / "data" / "original"
assert DATADIR.exists()

MODELS_DIR = CURDIR / "models"
assert MODELS_DIR.exists()

MODEL_ID = "facebook/bart-base"
MODEL_ID = MODELS_DIR / "BART_base_tune"

RESULT_MODEL_DIR = MODELS_DIR / "BART_base_tune"
if not RESULT_MODEL_DIR.exists():
    RESULT_MODEL_DIR.mkdir()

MAX_LENGTH = 512
DEVICE = "cuda"

In [3]:
df_train = pd.read_csv(DATADIR / "train.csv", index_col=0, sep="\t")
df_dev = pd.read_csv(DATADIR / "dev.csv", index_col=0, sep="\t")

In [4]:
tokenizer = BartTokenizer.from_pretrained(MODEL_ID)

In [5]:
model = BartForConditionalGeneration.from_pretrained(MODEL_ID).to("cuda")

In [6]:
df_train["sample"] = df_train.apply(lambda row: get_sample_from_row_original(row)[0], axis=1)
df_dev["sample"] = df_dev.apply(lambda row: get_sample_from_row_original(row)[0], axis=1)

df_train.shape, df_dev.shape

((2150060, 7), (255992, 7))

In [7]:
df_train.shape

(2150060, 7)

In [8]:
df_train = filter_irrelevant(df_train)
df_dev = filter_irrelevant(df_dev)

In [9]:
df_train.shape, df_dev.shape

((2135295, 7), (254996, 7))

In [10]:
df_train = prune_frequent_samples(df_train)  # здесь подрезаем несбалансированное начальное распределение
df_dev = df_dev.drop_duplicates(subset=["sample"])  # отсюда просто удаляем все дубли чтобы честно мериться

In [11]:
df_train.shape, df_dev.shape

((958668, 8), (52827, 7))

In [12]:
df_train["ln"] = df_train["sample"].map(len)
df_train = df_train[df_train["ln"] < 140]
df_train.shape, df_dev.shape

((958666, 9), (52827, 7))

In [13]:
df_train = df_train[["sample", "lemma"]]
df_dev = df_dev[["sample", "lemma"]]

In [14]:
train = Dataset.from_pandas(
    df_train[["sample", "lemma"]],
).rename_columns({
    "sample": "source",
    "lemma": "target",
}).shuffle(seed=42)

In [15]:
def tokenize_function(examples,):

    model_inputs = tokenizer(
        examples["source"],
        max_length=70,
        truncation=True,
        padding=False,
    )

    labels = tokenizer(
        examples["target"],
        max_length=70,
        truncation=True,
        padding=False,
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [16]:
tokenized_train = train.map(
    tokenize_function,
    batched=True,
    batch_size=1000,
    remove_columns=train.column_names,
)

Map: 100%|██████████| 958666/958666 [02:25<00:00, 6589.79 examples/s]


In [17]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
    label_pad_token_id=-100,
)

In [18]:
BATCH_SIZE = 56  # 56? 48

In [19]:
train_dataloader = DataLoader(
    tokenized_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator,
    num_workers=1,
    pin_memory=True,
    drop_last=False,
)

In [20]:
def train_batch(
    model: BartForConditionalGeneration,
    batch,
    optimizer,
):

    input_ids = batch['input_ids']
    attention_mask = batch['attention_mask']
    labels = batch['labels']

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )
    loss = outputs.loss

    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    optimizer.zero_grad()

    return loss

In [21]:
def train_epoch(
    model: BartForConditionalGeneration,
    iterator: DataLoader,
    optimizer,
    scheduler,
    device=DEVICE,
):

    model.eval()

    losses = []

    pbar = tqdm(iterator, desc="Training", unit="bs")

    for batch_id, batch in enumerate(pbar):
        batch = {k: val.to(device) for k, val in batch.items()}
        loss = train_batch(model, batch, optimizer)
        loss_item = loss.item()
        losses.append(loss_item)

        if batch_id % 100 == 0:
            pbar.set_description(f"Training (loss: {np.mean(losses):.6f})")
    scheduler.step()

    return np.mean(losses)

In [22]:
@torch.no_grad()
def validate(
    model: BartForConditionalGeneration,
    tokenizer:BartTokenizer,
    df: pd.DataFrame,
    device=DEVICE,
):
    model.eval()

    preds = predict(
        df["sample"].tolist(),
        model=model,
        tokenizer=tokenizer,
        device=device,
    )
    targets = df["lemma"].tolist()

    return lemmatization_accuracy(targets, preds)

In [29]:
optimizer = torch.optim.SGD(model.parameters(), lr=3e-2)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.65)

In [30]:
scheduler._last_lr

[0.03]

In [31]:
scheduler.step()
scheduler._last_lr

[0.0195]

In [32]:
scheduler.step()
scheduler._last_lr

[0.012675]

In [25]:
raise ZeroDivisionError

ZeroDivisionError: 

In [33]:
EPOCHS = 7

In [34]:
val_acc = validate(
    model,
    tokenizer,
    df_dev,
    DEVICE, 
)

print(f"Val Acc: {val_acc:.4f}")

1651it [03:31,  7.81it/s]                          

Val Acc: 0.9684


In [35]:
highest_val_acc = 0.5
patience = 0
train_losses = []
val_losses = []

max_patience = 2

for epoch in range(1, EPOCHS+1):

    train_loss = train_epoch(
        model,
        train_dataloader,
        optimizer,
        scheduler,
        DEVICE,
    )

    print(f"Train loss: {train_loss:.4f}")

    val_acc = validate(
        model,
        tokenizer,
        df_dev,
        DEVICE, 
    )

    print(f"Val Acc: {val_acc:.4f}")

    if val_acc > highest_val_acc:
        model.save_pretrained(RESULT_MODEL_DIR)
    else:
        patience += 1
        print(f"patience {patience}/{max_patience}")
        if patience == max_patience:
            print("Early Stopping!")
            break  # 0.002609

Training (loss: 0.002837): 100%|██████████| 17120/17120 [1:06:46<00:00,  4.27bs/s]


Train loss: 0.0028


1651it [03:12,  8.57it/s]                          


Val Acc: 0.9688


Training (loss: 0.002202):   4%|▍         | 759/17120 [02:57<1:03:52,  4.27bs/s]


KeyboardInterrupt: 

In [ ]:
# Training (loss: 0.021901): 100%|██████████| 17120/17120 [1:06:51<00:00,  4.27bs/s]
# Train loss: 0.0219
# 1651it [03:19,  8.28it/s]                          
# Val Acc: 0.9558
# Training (loss: 0.005300): 100%|██████████| 17120/17120 [1:06:53<00:00,  4.27bs/s]
# Train loss: 0.0053
# 1651it [03:14,  8.49it/s]                          
# Val Acc: 0.9652
# Training (loss: 0.003507): 100%|██████████| 17120/17120 [1:06:54<00:00,  4.26bs/s]
# Train loss: 0.0035
# 1651it [03:14,  8.48it/s]                          
# Val Acc: 0.9684




In [ ]:
scheduler._last_lr

[0.008238750000000001]

In [ ]:
# ep2 3500/17120; loss = 0.005856

In [ ]:
RESULT_MODEL_DIR

In [ ]:
tokenizer.save_pretrained(RESULT_MODEL_DIR)

('/home/smertlove/sandbox/hse/DEEPlom/FasterRubic2/models/BART_base_tune/tokenizer_config.json',
 '/home/smertlove/sandbox/hse/DEEPlom/FasterRubic2/models/BART_base_tune/special_tokens_map.json',
 '/home/smertlove/sandbox/hse/DEEPlom/FasterRubic2/models/BART_base_tune/vocab.json',
 '/home/smertlove/sandbox/hse/DEEPlom/FasterRubic2/models/BART_base_tune/merges.txt',
 '/home/smertlove/sandbox/hse/DEEPlom/FasterRubic2/models/BART_base_tune/added_tokens.json')